In [1]:
import torch
from torch.utils.data import DataLoader
from torch import nn
import torch.optim as optim
import torchvision.models as models

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'functions')))
from dataset import ChestXrayDataset
from train import train
from evaluation import plot_results ,eval_on_metrics
from gradcam import get_heatmap_for_resnet

In [2]:
IMAGE_PATH = "../archive/"
import glob

# Tüm alt klasörlerdeki jpg ve png dosyalarını alalım
image_paths = glob.glob(IMAGE_PATH + "**/images/*.[jp][pn]g", recursive=True)

print(f"Toplam {len(image_paths)} resim bulundu.")

Toplam 112120 resim bulundu.


In [3]:
TRAIN_PATH = '../data/AP_PA_Train.xlsx'
TEST_PATH = '../data/AP_PA_Test.xlsx'
VAL_PATH = '../data/AP_PA_Validation.xlsx'

In [4]:
num_classes = 2
EPOCHS = 30

In [5]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [6]:
train_dataset = ChestXrayDataset(TRAIN_PATH, image_paths,transform=transform)
val_dataset = ChestXrayDataset(TEST_PATH,image_paths, transform=transform)
test_dataset = ChestXrayDataset(VAL_PATH,image_paths, transform=transform)
print("Train size : ",len(train_dataset))
print("Validation size : ",len(val_dataset))
print("Test size : ",len(test_dataset))

Train size :  78566
Validation size :  16491
Test size :  17063


In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=48, shuffle=True,num_workers=18, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=48, shuffle=False,num_workers=18, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=48, shuffle=False,num_workers=10)

In [8]:
from fastai.data.core import DataLoaders

dls = DataLoaders(train_dataloader, val_dataloader)
dls.c = 2

In [9]:
from fastai.vision.all import *
import optuna

# 🔍 Optuna objective function
def objective(trial):
    # Hiperparametreleri seç
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    wd = trial.suggest_loguniform("weight_decay", 1e-5, 1e-2)
    dropout = trial.suggest_uniform("dropout", 0.1, 0.5)
    opt_func = trial.suggest_categorical("optimizer", [Adam, SGD])

    # Learner oluştur
    learn = Learner(
        dls,
        resnet50(num_classes=2),  # dls.c = 2
        loss_func=nn.CrossEntropyLoss(),
        opt_func=opt_func,
        metrics=accuracy,
        wd=wd
    )

    with learn.no_logging():
        learn.fit_one_cycle(1, lr_max=lr)

    acc = learn.validate()[1]
    return acc  # maximize accuracy

# 🔁 Optuna Study başlat
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

# 📊 En iyi sonucu yazdır
print("En iyi hiperparametreler:", study.best_params)
print("En iyi doğruluk:", study.best_value)
 

d:\anaconda\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-07-28 21:53:47,631] A new study created in memory with name: no-name-8f685b1e-26c7-4cca-b2cb-d2fccf2825ad
C:\Users\Furkan\AppData\Local\Temp\ipykernel_9612\2247076229.py:7: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
C:\Users\Furkan\AppData\Local\Temp\ipykernel_9612\2247076229.py:8: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  wd = trial.suggest_loguniform("weight_decay", 1

[I 2025-07-28 22:00:26,991] Trial 0 finished with value: 0.8165665864944458 and parameters: {'lr': 0.0006347476034704224, 'weight_decay': 0.00025164799362229276, 'dropout': 0.36881042238309225, 'optimizer': <function SGD at 0x000001309ED82980>}. Best is trial 0 with value: 0.8165665864944458.


[I 2025-07-28 22:07:12,088] Trial 1 finished with value: 0.9930264949798584 and parameters: {'lr': 7.580540416211198e-05, 'weight_decay': 0.00011576190885015582, 'dropout': 0.3454317955583531, 'optimizer': <function Adam at 0x000001309ED82D40>}. Best is trial 1 with value: 0.9930264949798584.


[I 2025-07-28 22:13:53,964] Trial 2 finished with value: 0.9935116171836853 and parameters: {'lr': 0.00013062977258990215, 'weight_decay': 1.3452479419286702e-05, 'dropout': 0.46251491900545205, 'optimizer': <function Adam at 0x000001309ED82D40>}. Best is trial 2 with value: 0.9935116171836853.


[I 2025-07-28 22:20:22,785] Trial 3 finished with value: 0.9240797758102417 and parameters: {'lr': 0.0012800904817507359, 'weight_decay': 1.1448753852650157e-05, 'dropout': 0.3789402957106919, 'optimizer': <function SGD at 0x000001309ED82980>}. Best is trial 2 with value: 0.9935116171836853.


...

[I 2025-07-28 22:39:35,569] Trial 6 finished with value: 0.9350554943084717 and parameters: {'lr': 0.0013227939572499929, 'weight_decay': 0.0022799806423286074, 'dropout': 0.13234944076189695, 'optimizer': <function SGD at 0x000001309ED82980>}. Best is trial 2 with value: 0.9935116171836853.


[I 2025-07-28 22:46:17,323] Trial 7 finished with value: 0.9940573573112488 and parameters: {'lr': 0.0007295942451365094, 'weight_decay': 2.7878781494437065e-05, 'dropout': 0.19356396606087825, 'optimizer': <function Adam at 0x000001309ED82D40>}. Best is trial 7 with value: 0.9940573573112488.


[I 2025-07-28 22:53:07,764] Trial 8 finished with value: 0.9905402660369873 and parameters: {'lr': 3.448616138310555e-05, 'weight_decay': 1.621386999968893e-05, 'dropout': 0.46773772229479726, 'optimizer': <function Adam at 0x000001309ED82D40>}. Best is trial 7 with value: 0.9940573573112488.


[I 2025-07-28 22:59:52,155] Trial 9 finished with value: 0.9889030456542969 and parameters: {'lr': 2.441775158491021e-05, 'weight_decay': 1.464846579940559e-05, 'dropout': 0.35683166942600675, 'optimizer': <function Adam at 0x000001309ED82D40>}. Best is trial 7 with value: 0.9940573573112488.


En iyi hiperparametreler: {'lr': 0.0007295942451365094, 'weight_decay': 2.7878781494437065e-05, 'dropout': 0.19356396606087825, 'optimizer': <function Adam at 0x000001309ED82D40>}
En iyi doğruluk: 0.9940573573112488


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
preds, targs = learn.get_preds(dl=test_dataloader)
pred_classes = preds.argmax(dim=1)

y_true = targs.numpy()
y_pred = pred_classes.numpy()
y_scores = preds[:,1].numpy()  # pozitif sınıf olasılığı

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC:       {roc_auc:.4f}")

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
train_losses = [x.item() for x in learn.recorder.losses]
# Not: learn.recorder.losses, batch bazında kayıp tutar, epoch bazında değil
val_losses = learn.recorder.values
val_accuracies = [x[1] for x in val_losses]  # validasyon accuracy sütunu (genelde 2. sütun)

# epoch sayısı
epochs = len(val_accuracies)

# Örnek çizim (train loss yerine validasyon kaybı çizilebilir)
plt.plot(range(epochs), val_accuracies, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()